<div style="background:linear-gradient(135deg,#4c0519 0%,#be123c 55%,#fb7185 100%);border-radius:18px;padding:30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fecdd3;font-weight:700;text-transform:uppercase">Chapter 65 · Solutions</div>
  <div style="font-size:30px;font-weight:900;margin:8px 0 4px">Study Design &amp; Data Quality — Worked Solutions ✅</div>
  <div style="font-size:14px;color:#fff1f2;max-width:720px;line-height:1.6">Five challenges, each verified in code.</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(650)

<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 1</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Spurious correlation from a confounder</div>
<div style="color:#4a5578;margin-top:5px">Let a confounder Z drive both X and Y. Show X and Y correlate, then control for Z and watch it vanish.</div>
</div>

In [2]:
Z = rng.uniform(0,10,2000)
X = 2*Z + rng.normal(0,2,2000)
Y = 3*Z + rng.normal(0,3,2000)
def resid(v): return v - np.poly1d(np.polyfit(Z,v,1))(Z)
print(f"raw corr(X,Y)            = {np.corrcoef(X,Y)[0,1]:.2f}")
print(f"corr after removing Z    = {np.corrcoef(resid(X),resid(Y))[0,1]:.2f}")

raw corr(X,Y)            = 0.90
corr after removing Z    = 0.05


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 2</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Randomization balances a confounder</div>
<div style="color:#4a5578;margin-top:5px">Compare the treated/control means of a covariate under self-selection vs a coin-flip assignment.</div>
</div>

In [3]:
n=5000; age=rng.uniform(20,70,n)
self_sel = rng.random(n) < 1/(1+np.exp(-(age-45)/8))
rand     = rng.random(n) < 0.5
print(f"self-selected: treated age {age[self_sel].mean():.0f} vs control {age[~self_sel].mean():.0f}  (imbalanced)")
print(f"randomized   : treated age {age[rand].mean():.0f} vs control {age[~rand].mean():.0f}  (balanced)")

self-selected: treated age 54 vs control 36  (imbalanced)
randomized   : treated age 45 vs control 45  (balanced)


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 3</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Build your own Simpson's paradox</div>
<div style="color:#4a5578;margin-top:5px">Construct two groups where Treatment X beats Y in each subgroup but loses overall.</div>
</div>

In [4]:
# X: great on easy cases but mostly given hard cases; Y: the reverse
rows = pd.DataFrame({"trt":["X","X","Y","Y"],"grp":["easy","hard","easy","hard"],
                     "succ":[19,41,85,5],"tot":[20,80,100,10]})
rows["rate"]=rows.succ/rows.tot
print(rows.to_string(index=False))
ov = rows.groupby("trt").apply(lambda g:g.succ.sum()/g.tot.sum(), include_groups=False)
print(f"\neasy: X {19/20:.0%} vs Y {85/100:.0%}; hard: X {41/80:.1%} vs Y {5/10:.0%}")
print(f"overall: X {ov['X']:.0%} vs Y {ov['Y']:.0%}  -> reversal!")

trt  grp  succ  tot   rate
  X easy    19   20 0.9500
  X hard    41   80 0.5125
  Y easy    85  100 0.8500
  Y hard     5   10 0.5000

easy: X 95% vs Y 85%; hard: X 51.2% vs Y 50%
overall: X 60% vs Y 82%  -> reversal!


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 4</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Data quality audit</div>
<div style="color:#4a5578;margin-top:5px">Audit a messy dataframe for completeness, uniqueness, and validity.</div>
</div>

In [5]:
df = pd.DataFrame({"id":[1,2,2,4],"score":[88, np.nan, 75, 150]})
print(f"completeness (score)   : {1-df.score.isna().mean():.0%}")
print(f"uniqueness (ids)       : {df.id.nunique()/len(df):.0%}")
print(f"validity (score 0-100) : {df.score.between(0,100).mean():.0%}")

completeness (score)   : 75%
uniqueness (ids)       : 75%
validity (score 0-100) : 50%


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 5</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Bias is a design flaw, not a sample-size one</div>
<div style="color:#4a5578;margin-top:5px">Show a confounded estimate stays biased as n grows from 2,000 to 50,000.</div>
</div>

In [6]:
for n in [2000, 50000]:
    age=rng.uniform(20,70,n); base=100-0.5*age
    tr = rng.random(n) < 1/(1+np.exp(-(age-45)/8))
    out = base + 8.0*tr + rng.normal(0,5,n)
    print(f"n={n:>6}: naive estimate {out[tr].mean()-out[~tr].mean():+.1f}  (true 8.0)")
print("bias constant: more data cannot fix a flawed design")

n=  2000: naive estimate -1.0  (true 8.0)
n= 50000: naive estimate -1.1  (true 8.0)
bias constant: more data cannot fix a flawed design


---
<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:10px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>